In [ ]:
import importlib
import ms_fl_scraper
ms_fl_scraper = importlib.reload(ms_fl_scraper)
scrape_section_url_async = ms_fl_scraper.scrape_section_url_async

import asyncio
import threading
import sys
import os

In [ ]:
def jsonl_has_record(path):
    if not os.path.exists(path):
        return False
    if os.path.getsize(path) == 0:
        return False
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    return True
    except Exception:
        return False
    return False

def run_scraper(url, state, output_file):
    if sys.platform == "win32":
        loop = asyncio.ProactorEventLoop()
        asyncio.set_event_loop(loop)
    else:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    
    try:
        loop.run_until_complete(scrape_section_url_async(
            section_url=url,
            state=state,
            output_file=output_file,
            require_complete_tree=True
        ))
        # treat no-record files as failed scrape
        if not jsonl_has_record(output_file):
            if os.path.exists(output_file):
                os.remove(output_file)
            return False
        return True
    except Exception as e:
        print(f"[SCRAPE FAILED] {state} {url}: {e}")
        if os.path.exists(output_file) and not jsonl_has_record(output_file):
            os.remove(output_file)
        return False
    finally:
        loop.close()

In [ ]:
#t = threading.Thread(target=run_scraper)
#t.start()
#t.join()

cool. können wir jetzt ein loop schreiben? das soll über usstates50.xlsx gehen und für alle staaten die spalten cn_source, el_source el2_source und le_source bearbeiten, sofern dort ein link zu findlaw drin ist und den scraper aufrufen. alle output jsonl files sollen in staats-spezifischen unterordnern in einem neuen ordner "us_codes" landen. passt?

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
import os
import shutil

In [ ]:
# we want to collect all links in usstates50xlsx. cols cn_source, el_source, el2_source, le_source if they refer to findlaw pages and put them in a long format dataframe with columns state, type, url; type should be the column name where the url was found, e.g. cn_source, el_source, el2_source, le_source but without _source

us = pd.read_excel("usstates50.xlsx")
links = []
for index, row in us.iterrows():
    for col in ["cn_source", "el_source", "el2_source", "le_source"]:
        url = row[col]
        if isinstance(url, str) and "findlaw" in url:
            links.append({
                "state": row["state"],
                "type": col.rstrip("_source"),
                "url": url
            })
links_df = pd.DataFrame(links)

# create list of all unique urls

urls = links_df["url"].unique()

In [ ]:
## scrape all links in loop
## all scraped urls are stored in directory state_codes/STATE/type.jsonl
## IMPORTANT: iterate by (state, type, url), not only by url,
## so Louisiana el/l can be scraped separately even when they share the same source URL

tasks_df = links_df.drop_duplicates(subset=["state", "type", "url"]).reset_index(drop=True)

for _, task in tqdm(tasks_df.iterrows(), total=len(tasks_df)):
    state = task["state"]
    type = task["type"]
    url = task["url"]

    # if directory does not exist create it
    if not os.path.exists(f"state_codes/{state}"):
        os.makedirs(f"state_codes/{state}")
    output_file = f"state_codes/{state}/{type}.jsonl"

    # skip only when we already have a valid non-empty jsonl with at least one record
    if jsonl_has_record(output_file):
        continue
    if os.path.exists(output_file):
        os.remove(output_file)

    try:
        t = threading.Thread(target=run_scraper, args=(url, state, output_file))
        t.start()
        t.join()
    except Exception as e:
        print(f"[LOOP ERROR] {state} {type} {url}: {e}")
        if os.path.exists(output_file):
            os.remove(output_file)
        continue

    # if scraping failed or produced no records, remove artifact and continue
    if not jsonl_has_record(output_file):
        if os.path.exists(output_file):
            os.remove(output_file)
        continue

    # copy only to exact same type duplicates of same URL (never cross-type copy)
    dup_rows = links_df[links_df["url"] == url]
    if len(dup_rows) > 1:
        for _, row in dup_rows.iterrows():
            other_state = row["state"]
            other_type = row["type"]

            if other_state == state and other_type == type:
                continue
            if other_type != type:
                continue

            other_output_file = f"state_codes/{other_state}/{other_type}.jsonl"
            if not os.path.exists(f"state_codes/{other_state}"):
                os.makedirs(f"state_codes/{other_state}")
            shutil.copyfile(output_file, other_output_file)